In [111]:
%pwd

'C:\\Users\\Soumita\\Downloads\\Sucheta\\Text-summarizer'

In [112]:
import os
os.chdir("C:\\Users\\Soumita\\Downloads\\Sucheta\\Text-summarizer")

In [113]:
%pwd

'C:\\Users\\Soumita\\Downloads\\Sucheta\\Text-summarizer'

In [114]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen = True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [115]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [116]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):


            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)

            create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config
    

In [117]:
import os
import urllib.request as request
import zipfile
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

In [118]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  

        
    
    import zipfile
    import os

    def extract_zip_file(self):

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        if not zipfile.is_zipfile(self.config.local_data_file):
            raise Exception(f"{self.config.local_data_file} is not a valid zip file")

        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [119]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-05-12 18:05:59,711: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-12 18:05:59,716: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-12 18:05:59,720: INFO: common: created directory at: artifacts]
[2026-05-12 18:05:59,725: INFO: common: created directory at: artifacts/data_ingestion]
[2026-05-12 18:06:10,412: INFO: 1086511670: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 23855247
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "e6e1a4cb835dd3d5a29771ff3f39621043d7ad0e9f898221c3ff760d2545d1b2"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 736A:254320:11F2CF:2D51C2:6A031EAF
Accept-Ranges: bytes
Date: Tue, 12 May 2026 12:36:02 GMT
Via: 1.1 varnish
X-Served-By: cache-ccu830047-CCU
X-Cache: